In [1]:

# SHARED-BACKBONE CNN + FEATURE EXTRACTION
# Run in your existing TensorFlow 2.21 / GPU-enabled Jupyter kernel.

import os, csv, json, time
from pathlib import Path
import numpy as np
import tensorflow as tf

ROOT = Path.home() / "sharedcnn_project"
DATA = ROOT / "data"
OUT = ROOT / "runs"
FEAT = ROOT / "features"
OUT.mkdir(parents=True, exist_ok=True)
FEAT.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (128, 128)
BATCH = 16
MAX_EPOCHS = 50
PATIENCE = 5
SEED = 42
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

tf.keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Dataset root:", DATA)

FOOD_DIR, WATER_DIR = DATA / "food", DATA / "water"
FOOD_CLASSES = sorted(
    p.name for p in FOOD_DIR.iterdir()
    if p.is_dir() and p.name.startswith(("fresh_", "stale_"))
)
WATER_CLASSES = ["Clear", "Foam", "Highly_Turbid", "Medium_Turbid"]

assert len(FOOD_CLASSES) == 12, f"Expected 12 food folders; found {FOOD_CLASSES}"
assert all((WATER_DIR / s).is_dir() for s in ("train", "valid", "test"))

# Food: stratified file-level split. This is preliminary if duplicate/augmented
# versions of the same original image exist.
from sklearn.model_selection import train_test_split

food_paths, food_labels = [], []
for label, cls in enumerate(FOOD_CLASSES):
    for p in sorted((FOOD_DIR / cls).rglob("*")):
        if p.is_file() and p.suffix.lower() in EXTS:
            food_paths.append(str(p))
            food_labels.append(label)

food_labels = np.asarray(food_labels, dtype=np.int32)
indices = np.arange(len(food_paths))
train_ids, temp_ids = train_test_split(
    indices, test_size=.30, random_state=SEED, stratify=food_labels
)
val_ids, test_ids = train_test_split(
    temp_ids, test_size=.50, random_state=SEED,
    stratify=food_labels[temp_ids]
)

def load_path(path, label):
    image = tf.io.decode_image(
        tf.io.read_file(path), channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    return image, tf.cast(label, tf.int32)

def food_ds(ids, training=False):
    paths = np.asarray(food_paths)[ids]
    labels = food_labels[ids]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(ids), 4000), seed=SEED)
    return ds.map(load_path, num_parallel_calls=tf.data.AUTOTUNE) \
             .batch(BATCH).prefetch(tf.data.AUTOTUNE)

food_train = food_ds(train_ids, True)
food_val = food_ds(val_ids)
food_test = food_ds(test_ids)

def water_ds(split, training=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        WATER_DIR / split,
        class_names=WATER_CLASSES,
        image_size=IMG_SIZE,
        batch_size=BATCH,
        shuffle=training,
        seed=SEED,
    )
    return ds.prefetch(tf.data.AUTOTUNE)

water_train = water_ds("train", True)
water_val = water_ds("valid")
water_test = water_ds("test")

print("Food:", len(food_paths), "images;", len(FOOD_CLASSES), "classes")
print("Food split sizes:", len(train_ids), len(val_ids), len(test_ids))
print("Water splits loaded; unlabeled images excluded.")

def build_model(kind):
    inputs = tf.keras.Input((*IMG_SIZE, 3), name="image")

    if kind == "mobilenetv2":
        base = tf.keras.applications.MobileNetV2(
            input_shape=(*IMG_SIZE, 3),
            include_top=False,
            weights="imagenet",
        )
        base.trainable = False
        x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
        x = base(x, training=False)
        pooled = tf.keras.layers.GlobalAveragePooling2D(name="pooled_features")(x)
    else:
        x = tf.keras.layers.Rescaling(1./255)(inputs)
        for filters in (16, 32, 64, 128):
            x = tf.keras.layers.Conv2D(
                filters, 3, padding="same", use_bias=False
            )(x)
            x = tf.keras.layers.BatchNormalization()(x)
            x = tf.keras.layers.ReLU()(x)
            x = tf.keras.layers.MaxPooling2D()(x)
        pooled = tf.keras.layers.GlobalAveragePooling2D(name="pooled_features")(x)

    embedding = tf.keras.layers.Dense(
        128, activation="relu", name="embedding_128"
    )(pooled)

    food_head = tf.keras.layers.Dense(
        len(FOOD_CLASSES), activation="softmax", name="food_head"
    )(embedding)
    water_head = tf.keras.layers.Dense(
        len(WATER_CLASSES), activation="softmax", name="water_head"
    )(embedding)

    model = tf.keras.Model(inputs, [food_head, water_head], name=f"{kind}_shared")
    embedder = tf.keras.Model(
        inputs, [embedding, pooled], name=f"{kind}_feature_extractor"
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=[
            "sparse_categorical_crossentropy",
            "sparse_categorical_crossentropy",
        ],
        metrics=[["accuracy"], ["accuracy"]],
    )
    return model, embedder

def xy(ds):
    for images, labels in ds:
        yield images, labels

def evaluate_domain(model, ds, domain_index):
    losses, accs = [], []
    for images, labels in ds:
        outputs = model(images, training=False)
        logits = outputs[domain_index]
        loss = tf.keras.losses.sparse_categorical_crossentropy(labels, logits)
        pred = tf.argmax(logits, axis=1, output_type=tf.int32)
        losses.append(float(tf.reduce_mean(loss)))
        accs.append(float(tf.reduce_mean(tf.cast(pred == labels, tf.float32))))
    return float(np.mean(losses)), float(np.mean(accs))

def train_shared(kind):
    run_dir = OUT / f"{kind}_shared"
    run_dir.mkdir(parents=True, exist_ok=True)
    model, _ = build_model(kind)
    best_val, wait = float("inf"), 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        start = time.time()
        # Alternate domain batches through the same model/backbone.
        food_losses, water_losses = [], []
        food_accs, water_accs = [], []

        for images, labels in food_train:
            loss = model.train_on_batch(
                images,
                [labels, tf.zeros_like(labels)],
                return_dict=True,
            )
            # Only the food head is supervised on this update.
            food_losses.append(float(loss["food_head_loss"]))
            food_accs.append(float(loss.get("food_head_accuracy", 0.0)))

        for images, labels in water_train:
            loss = model.train_on_batch(
                images,
                [tf.zeros_like(labels), labels],
                return_dict=True,
            )
            # Only the water head is supervised on this update.
            water_losses.append(float(loss["water_head_loss"]))
            water_accs.append(float(loss.get("water_head_accuracy", 0.0)))

        fv_loss, fv_acc = evaluate_domain(model, food_val, 0)
        wv_loss, wv_acc = evaluate_domain(model, water_val, 1)
        val_loss = fv_loss + wv_loss

        row = {
            "epoch": epoch,
            "food_train_loss": float(np.mean(food_losses)),
            "food_val_loss": fv_loss,
            "food_val_accuracy": fv_acc,
            "water_train_loss": float(np.mean(water_losses)),
            "water_val_loss": wv_loss,
            "water_val_accuracy": wv_acc,
            "seconds": time.time() - start,
        }
        history.append(row)
        print(row, flush=True)

        model.save(run_dir / "latest.keras")
        if val_loss < best_val:
            best_val, wait = val_loss, 0
            model.save(run_dir / "best.keras")
        else:
            wait += 1
            if wait >= PATIENCE:
                print("Early stopping at epoch", epoch, flush=True)
                break

        with open(run_dir / "history.json", "w") as f:
            json.dump(history, f, indent=2)

    # Reload the best validation checkpoint before testing/exporting.
    model = tf.keras.models.load_model(run_dir / "best.keras")
    food_test_metrics = evaluate_domain(model, food_test, 0)
    water_test_metrics = evaluate_domain(model, water_test, 1)
    print("Food test loss/accuracy:", food_test_metrics)
    print("Water test loss/accuracy:", water_test_metrics)

    model.save(run_dir / "final.keras")
    return model, run_dir

def export_embeddings(model, kind, domain, paths, labels, class_names):
    folder = FEAT / f"{kind}_shared" / domain
    folder.mkdir(parents=True, exist_ok=True)
    embedder = tf.keras.Model(
        model.input,
        [
            model.get_layer("embedding_128").output,
            model.get_layer("pooled_features").output,
        ],
    )

    all_emb, all_pooled = [], []
    for start in range(0, len(paths), BATCH):
        batch_paths = paths[start:start+BATCH]
        images = []
        for path in batch_paths:
            im = tf.io.decode_image(
                tf.io.read_file(path), channels=3, expand_animations=False
            )
            im.set_shape([None, None, 3])
            images.append(tf.image.resize(im, IMG_SIZE))
        batch = tf.stack(images)
        emb, pooled = embedder(batch, training=False)
        all_emb.append(emb.numpy())
        all_pooled.append(pooled.numpy())

    emb = np.concatenate(all_emb)
    pooled = np.concatenate(all_pooled)
    np.save(folder / "emb_128.npy", emb)
    np.save(folder / "pooled.npy", pooled)
    with open(folder / "index.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["path", "label_id", "label_name"])
        for path, label in zip(paths, labels):
            writer.writerow([path, int(label), class_names[int(label)]])
    print(f"Exported {domain}: embeddings={emb.shape}, pooled={pooled.shape}")

# Train shared MobileNetV2 and custom CNN.
for kind in ("mobilenetv2", "customcnn"):
    print("\n\n========== TRAINING", kind, "==========", flush=True)
    model, run_dir = train_shared(kind)

    export_embeddings(
        model, kind, "food",
        [food_paths[i] for i in train_ids],
        food_labels[train_ids], FOOD_CLASSES,
    )

    water_paths, water_labels = [], []
    for label, cls in enumerate(WATER_CLASSES):
        for p in sorted((WATER_DIR / "train" / cls).rglob("*")):
            if p.is_file() and p.suffix.lower() in EXTS:
                water_paths.append(str(p))
                water_labels.append(label)
    export_embeddings(
        model, kind, "water",
        water_paths, water_labels, WATER_CLASSES,
    )

print("\nFinished shared training and embedding exports.")
print("Models:", OUT)
print("Features:", FEAT)


I0000 00:00:1792715544.034968    8760 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792715544.154693    8760 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792715546.440818    8760 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.22.0-rc0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Dataset root: /home/kapoo/sharedcnn_project/data


I0000 00:00:1792715553.618847    8760 gpu_device.cc:2036] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3536 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Found 617 files belonging to 4 classes.
Found 159 files belonging to 4 classes.
Found 377 files belonging to 4 classes.
Food: 14682 images; 12 classes
Food split sizes: 10277 2202 2203
Water splits loaded; unlabeled images excluded.


========== TRAINING mobilenetv2 ==========
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


I0000 00:00:1792715563.763461    8760 service.cc:154] XLA service 0x5935390ba730 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1792715563.763512    8760 service.cc:170]   StreamExecutor [0]: NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9 (Driver: 13.1.0[592.82.0]; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1792715563.824156    8760 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1792715563.340701    8760 cuda_dnn.cc:442] Loaded cuDNN version 92700
W0000 00:00:1792715570.203033    8981 cuda_timer.cc:88] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1792715572.958462    8760 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
W0000 00:00:1792715572.995607    8

{'epoch': 1, 'food_train_loss': 0.1530092339753608, 'food_val_loss': 19.414377938146178, 'food_val_accuracy': 0.11503623188405797, 'water_train_loss': 1.6463284369648392, 'water_val_loss': 0.20361928927156897, 'water_val_accuracy': 0.9625, 'seconds': 80.16892457008362}
{'epoch': 2, 'food_train_loss': 0.1886003643349805, 'food_val_loss': 5.85878806874372, 'food_val_accuracy': 0.25036231892696326, 'water_train_loss': 0.47853223914698434, 'water_val_loss': 0.28155283977229006, 'water_val_accuracy': 0.9625, 'seconds': 52.469260931015015}
{'epoch': 3, 'food_train_loss': 0.06639012321178014, 'food_val_loss': 0.05745079426117666, 'food_val_accuracy': 0.980072463768116, 'water_train_loss': 0.11896090868611235, 'water_val_loss': 0.1453912051336374, 'water_val_accuracy': 0.98125, 'seconds': 54.59243726730347}
{'epoch': 4, 'food_train_loss': 0.015540765163544901, 'food_val_loss': 0.026440637802727946, 'food_val_accuracy': 0.9923007246376812, 'water_train_loss': 0.06094307456171522, 'water_val_los

W0000 00:00:1792716420.673696    8760 prefetch_dataset_op.cc:476] SEVERE STARVATION: Element UID 1792716408548308291 in buffer 'PrefetchDataset/_14' sat unconsumed for 12.121 seconds!


{'epoch': 1, 'food_train_loss': 0.7118808625738136, 'food_val_loss': 18.30548347943071, 'food_val_accuracy': 0.11503623188405797, 'water_train_loss': 1.8330847915166464, 'water_val_loss': 1.0596824226900936, 'water_val_accuracy': 0.6875, 'seconds': 66.34713053703308}
{'epoch': 2, 'food_train_loss': 0.5615143826054655, 'food_val_loss': 26.60568542756896, 'food_val_accuracy': 0.11503623188405797, 'water_train_loss': 1.870051988424399, 'water_val_loss': 1.5562617802526801, 'water_val_accuracy': 0.55625, 'seconds': 35.39691686630249}
{'epoch': 3, 'food_train_loss': 0.4300249046754689, 'food_val_loss': 12.345805230348006, 'food_val_accuracy': 0.12318840579710146, 'water_train_loss': 1.6475896741717289, 'water_val_loss': 1.542602265602909, 'water_val_accuracy': 0.55625, 'seconds': 35.29827284812927}
{'epoch': 4, 'food_train_loss': 0.35679946404130053, 'food_val_loss': 6.586189772771752, 'food_val_accuracy': 0.12047101449275362, 'water_train_loss': 1.5117729833492866, 'water_val_loss': 0.8164

In [2]:
import sys
print(sys.executable)

/home/kapoo/miniconda3/bin/python


In [3]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [4]:

import sys
import numpy as np
import tensorflow as tf

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


Python: /home/kapoo/miniconda3/bin/python
NumPy: 2.5.4
TensorFlow: 2.22.0-rc0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:

import subprocess
import sys

env_python = "/home/kapoo/miniconda3/envs/sharedcnn/bin/python"

# Install the missing kernel package in sharedcnn
subprocess.check_call([
    env_python, "-m", "pip", "install", "ipykernel"
])

# Register it with Jupyter
subprocess.check_call([
    env_python, "-m", "ipykernel", "install",
    "--user", "--name", "sharedcnn",
    "--display-name", "Python (sharedcnn)"
])

print("SUCCESS: Kernel registered.")
print("Next: Kernel → Change Kernel → Python (sharedcnn)")


  Using cached ipykernel-7.4.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached comm-0.2.3-py3-none-any.whl.metadata (3.7 kB)
  Using cached ipython-9.17.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached jupyter_client-8.10.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached matplotlib_inline-0.2.2-py3-none-any.whl.metadata (2.4 kB)
  Using cached nest_asyncio2-1.7.4-py3-none-any.whl.metadata (7.0 kB)
  Using cached pyzmq-27.2.0-cp312-abi3-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl.metadata (3.8 kB)
  Using cached tornado-6.6-cp311-abi3-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (2.6 kB)
  Using cached ipython_pygments_lexers-1.1.1-py3-none-any.whl.metadata (1.1 kB)
  Using cached jedi-0.20.1-py2.py3-none-any.whl.metadata (23 kB)
  Using cached pexpect-4.9.0-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached prompt_toolkit-3.0.53-py3-none-any.whl.metadata (6.4 kB)
  Using cached psutil-7.2.2-cp36-abi3-manylinux2010_x86_64.manylinux_2_12_x86_64.

In [7]:

import sys
import numpy as np
import tensorflow as tf

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


Python: /home/kapoo/miniconda3/bin/python
TensorFlow: 2.22.0-rc0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [8]:

import os
import sys
import glob
import numpy as np
import tensorflow as tf

ROOT = "/home/kapoo/sharedcnn_project"

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

print("\n=== SAVED MODELS ===")
for path in glob.glob(ROOT + "/runs/**/*", recursive=True):
    if os.path.isfile(path) and path.endswith((".keras", ".h5")):
        print(path, f"({os.path.getsize(path) / 1e6:.1f} MB)")

print("\n=== FEATURE FILES ===")
for path in glob.glob(ROOT + "/features/**/*", recursive=True):
    if os.path.isfile(path):
        print(path, f"({os.path.getsize(path) / 1e6:.1f} MB)")

print("\n=== OUTPUT DIRECTORIES ===")
for folder in ["runs", "features", "outputs", "data"]:
    path = os.path.join(ROOT, folder)
    print(folder, "exists:", os.path.isdir(path))


Python: /home/kapoo/miniconda3/bin/python
TensorFlow: 2.22.0-rc0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

=== SAVED MODELS ===
/home/kapoo/sharedcnn_project/runs/mobilenetv2_shared/best.keras (11.6 MB)
/home/kapoo/sharedcnn_project/runs/mobilenetv2_shared/latest.keras (11.6 MB)
/home/kapoo/sharedcnn_project/runs/mobilenetv2_shared/final.keras (11.6 MB)
/home/kapoo/sharedcnn_project/runs/customcnn_shared/best.keras (1.5 MB)
/home/kapoo/sharedcnn_project/runs/customcnn_shared/latest.keras (1.5 MB)
/home/kapoo/sharedcnn_project/runs/customcnn_shared/final.keras (1.5 MB)

=== FEATURE FILES ===
/home/kapoo/sharedcnn_project/features/mobilenetv2_shared/water/pooled.npy (3.2 MB)
/home/kapoo/sharedcnn_project/features/mobilenetv2_shared/water/index.csv (0.1 MB)
/home/kapoo/sharedcnn_project/features/mobilenetv2_shared/water/emb_128.npy (0.3 MB)
/home/kapoo/sharedcnn_project/features/mobilenetv2_shared/food/pooled.npy (52.6 MB)
/home/kapoo/sharedcnn_project/featu

In [9]:

import os, glob, json
import numpy as np
import tensorflow as tf

ROOT = "/home/kapoo/sharedcnn_project"

print("=== MODEL ARCHITECTURES ===")
for p in sorted(glob.glob(ROOT + "/runs/**/*.keras", recursive=True)):
    try:
        m = tf.keras.models.load_model(p, compile=False)
        print("\n", os.path.relpath(p, ROOT))
        print("Input:", m.input_shape)
        print("Outputs:", [(x.name, x.shape) for x in m.outputs])
    except Exception as e:
        print("\nCould not inspect:", p, str(e)[:200])

print("\n=== EMBEDDING SHAPES ===")
for p in sorted(glob.glob(ROOT + "/features/**/*.npy", recursive=True)):
    a = np.load(p, mmap_mode="r")
    print(os.path.relpath(p, ROOT), a.shape, a.dtype)

print("\n=== OTHER DELIVERABLES ===")
for pattern in ["**/meta.json", "**/*mapping*.json",
                "**/*metrics*.csv", "**/*history*.csv",
                "**/*confusion*.png", "**/*report*.txt"]:
    found = glob.glob(ROOT + "/" + pattern, recursive=True)
    print(pattern, "->", len(found))
    for f in found[:8]:
        print(" ", os.path.relpath(f, ROOT))


=== MODEL ARCHITECTURES ===

 runs/customcnn_shared/best.keras
Input: (None, 128, 128, 3)
Outputs: [('keras_tensor_593', (None, 12)), ('keras_tensor_595', (None, 4))]

 runs/customcnn_shared/final.keras
Input: (None, 128, 128, 3)
Outputs: [('keras_tensor_635', (None, 12)), ('keras_tensor_637', (None, 4))]

 runs/customcnn_shared/latest.keras
Input: (None, 128, 128, 3)
Outputs: [('keras_tensor_677', (None, 12)), ('keras_tensor_679', (None, 4))]

 runs/mobilenetv2_shared/best.keras
Input: (None, 128, 128, 3)
Outputs: [('keras_tensor_1007', (None, 12)), ('keras_tensor_1009', (None, 4))]

 runs/mobilenetv2_shared/final.keras
Input: (None, 128, 128, 3)
Outputs: [('keras_tensor_1337', (None, 12)), ('keras_tensor_1339', (None, 4))]

 runs/mobilenetv2_shared/latest.keras
Input: (None, 128, 128, 3)
Outputs: [('keras_tensor_1667', (None, 12)), ('keras_tensor_1669', (None, 4))]

=== EMBEDDING SHAPES ===
features/customcnn_shared/food/emb_128.npy (10277, 128) float32
features/customcnn_shared/food

In [10]:

import os, glob, json
import numpy as np
import tensorflow as tf

ROOT = "/home/kapoo/sharedcnn_project"

print("=== MODEL ARCHITECTURES ===")
for p in sorted(glob.glob(ROOT + "/runs/**/*.keras", recursive=True)):
    try:
        m = tf.keras.models.load_model(p, compile=False)
        print("\n", os.path.relpath(p, ROOT))
        print("Input:", m.input_shape)
        print("Outputs:", [(x.name, x.shape) for x in m.outputs])
    except Exception as e:
        print("\nCould not inspect:", p, str(e)[:200])

print("\n=== EMBEDDING SHAPES ===")
for p in sorted(glob.glob(ROOT + "/features/**/*.npy", recursive=True)):
    a = np.load(p, mmap_mode="r")
    print(os.path.relpath(p, ROOT), a.shape, a.dtype)

print("\n=== OTHER DELIVERABLES ===")
for pattern in ["**/meta.json", "**/*mapping*.json",
                "**/*metrics*.csv", "**/*history*.csv",
                "**/*confusion*.png", "**/*report*.txt"]:
    found = glob.glob(ROOT + "/" + pattern, recursive=True)
    print(pattern, "->", len(found))
    for f in found[:8]:
        print(" ", os.path.relpath(f, ROOT))


=== MODEL ARCHITECTURES ===

 runs/customcnn_shared/best.keras
Input: (None, 128, 128, 3)
Outputs: [('keras_tensor_1709', (None, 12)), ('keras_tensor_1711', (None, 4))]

 runs/customcnn_shared/final.keras
Input: (None, 128, 128, 3)
Outputs: [('keras_tensor_1751', (None, 12)), ('keras_tensor_1753', (None, 4))]

 runs/customcnn_shared/latest.keras
Input: (None, 128, 128, 3)
Outputs: [('keras_tensor_1793', (None, 12)), ('keras_tensor_1795', (None, 4))]

 runs/mobilenetv2_shared/best.keras
Input: (None, 128, 128, 3)
Outputs: [('keras_tensor_2123', (None, 12)), ('keras_tensor_2125', (None, 4))]

 runs/mobilenetv2_shared/final.keras
Input: (None, 128, 128, 3)
Outputs: [('keras_tensor_2453', (None, 12)), ('keras_tensor_2455', (None, 4))]

 runs/mobilenetv2_shared/latest.keras
Input: (None, 128, 128, 3)
Outputs: [('keras_tensor_2783', (None, 12)), ('keras_tensor_2785', (None, 4))]

=== EMBEDDING SHAPES ===
features/customcnn_shared/food/emb_128.npy (10277, 128) float32
features/customcnn_share

In [12]:

import os, glob, csv
import numpy as np

ROOT = "/home/kapoo/sharedcnn_project"

print("=== PROJECT SCRIPTS ===")
for p in sorted(set(glob.glob(ROOT + "/*.py") +
                    glob.glob(ROOT + "/**/*.py", recursive=True))):
    print(os.path.relpath(p, ROOT))

print("\n=== DATASET FOLDERS ===")
for base in ["data/food", "data/water"]:
    path = os.path.join(ROOT, base)
    print("\n" + base)
    if os.path.isdir(path):
        for current, dirs, files in os.walk(path):
            dirs.sort()
            imgs = [f for f in files if f.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp", ".webp")
            )]
            if imgs or current == path:
                print(os.path.relpath(current, ROOT),
                      "| images:", len(imgs),
                      "| subfolders:", dirs[:10])
    else:
        print("NOT FOUND")

print("\n=== FEATURE INDEX FILES ===")
for p in sorted(glob.glob(ROOT + "/features/**/index.csv",
                          recursive=True)):
    print("\n", os.path.relpath(p, ROOT))
    with open(p, newline="", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for _, row in zip(range(4), reader):
            print(row)

print("\n=== TRAINING HISTORY FILES ===")
for p in sorted(glob.glob(ROOT + "/outputs/*history*.csv")):
    print("\n", os.path.basename(p))
    with open(p, newline="", encoding="utf-8-sig") as f:
        rows = list(csv.reader(f))
    for row in rows[:1] + rows[-3:]:
        print(row)

print("\n=== ROOT FILES ===")
print(sorted(os.listdir(ROOT)))


=== PROJECT SCRIPTS ===
shared_train.py
train_models.py

=== DATASET FOLDERS ===

data/food
data/food | images: 0 | subfolders: ['fresh_apple', 'fresh_banana', 'fresh_bitter_gourd', 'fresh_capsicum', 'fresh_orange', 'fresh_tomato', 'stale_apple', 'stale_banana', 'stale_bitter_gourd', 'stale_capsicum']
data/food/fresh_apple | images: 1693 | subfolders: []
data/food/fresh_banana | images: 1581 | subfolders: []
data/food/fresh_bitter_gourd | images: 327 | subfolders: []
data/food/fresh_capsicum | images: 990 | subfolders: []
data/food/fresh_orange | images: 1466 | subfolders: []
data/food/fresh_tomato | images: 981 | subfolders: []
data/food/stale_apple | images: 2342 | subfolders: []
data/food/stale_banana | images: 1467 | subfolders: []
data/food/stale_bitter_gourd | images: 357 | subfolders: []
data/food/stale_capsicum | images: 901 | subfolders: []
data/food/stale_orange | images: 1595 | subfolders: []
data/food/stale_tomato | images: 982 | subfolders: []

data/water
data/water | imag

In [13]:

import os

ROOT = "/home/kapoo/sharedcnn_project"

for name in ["shared_train.py", "train_models.py"]:
    path = os.path.join(ROOT, name)
    print("\n" + "=" * 25, name, "=" * 25)
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # Print relevant sections only
    keywords = (
        "def ", "compile(", "fit(", "evaluate(", "predict(",
        "GlobalAveragePooling", "Dense(", "emb_128",
        "pooled", "confusion_matrix", "classification_report",
        "meta.json", "EarlyStopping", "epochs="
    )
    for i, line in enumerate(lines, 1):
        if any(k in line for k in keywords):
            print(f"{i:4}: {line.rstrip()}")



========================= shared_train.py =========================
  24: def collect_food():
  33: def load_image(path, label):
  41: def make_food_ds(paths, labels, training):
  48: def make_water_ds(split, training):
  59: def build_shared(architecture):
  69:         pooled = tf.keras.layers.GlobalAveragePooling2D()(x)
  77:         pooled = tf.keras.layers.GlobalAveragePooling2D()(x)
  79:     x = tf.keras.layers.Dropout(0.2)(pooled)
  80:     embedding = tf.keras.layers.Dense(128, activation="relu", name="embedding_128")(x)
  82:     food_head = tf.keras.layers.Dense(
  85:     water_head = tf.keras.layers.Dense(
  90:     embedder = tf.keras.Model(inputs, {"embedding": embedding, "pooled": pooled})
  92:     model.compile(
 140:         food_metrics = model.fit(
 143:             epochs=1,
 146:         water_metrics = model.fit(
 149:             epochs=1,

========================= train_models.py =========================
  22: def build(n):
  30:     x = tf.keras.layers.Glo

In [14]:

import os, glob, csv
import numpy as np

ROOT = "/home/kapoo/sharedcnn_project"
report = []

def add(s=""):
    report.append(str(s))
    print(s)

add("SHARED-BACKBONE CNN PROJECT AUDIT")
add("=" * 40)

# Model files
add("\nSAVED MODELS")
for p in sorted(glob.glob(ROOT + "/runs/**/*.keras", recursive=True)):
    add(f"{os.path.relpath(p, ROOT)} | {os.path.getsize(p)/1e6:.2f} MB")

# Feature validation
add("\nFEATURE ARRAY VALIDATION")
for p in sorted(glob.glob(ROOT + "/features/**/*.npy", recursive=True)):
    arr = np.load(p, mmap_mode="r")
    add(f"{os.path.relpath(p, ROOT)} | shape={arr.shape} | dtype={arr.dtype}")

# Index row counts
add("\nFEATURE INDEX COUNTS")
for p in sorted(glob.glob(ROOT + "/features/**/index.csv", recursive=True)):
    with open(p, newline="", encoding="utf-8-sig") as f:
        rows = sum(1 for _ in csv.reader(f)) - 1
    add(f"{os.path.relpath(p, ROOT)} | rows={rows}")

# Script configuration
add("\nTRAINING CONFIGURATION")
for name in ["shared_train.py", "train_models.py"]:
    path = os.path.join(ROOT, name)
    add(f"\n--- {name} ---")
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    for i, line in enumerate(lines, 1):
        if any(k in line for k in [
            "EPOCHS =", "BATCH =", "IMG_SIZE", "SEED =",
            "def build(", "def build_shared", "EarlyStopping",
            "ModelCheckpoint", "epochs=", "train_test_split",
            "meta.json", "class_names"
        ]):
            add(f"{i}: {line.rstrip()}")

out = os.path.join(ROOT, "project_audit.txt")
with open(out, "w", encoding="utf-8") as f:
    f.write("\n".join(report))

print("\nSaved report:", out)


SHARED-BACKBONE CNN PROJECT AUDIT

SAVED MODELS
runs/customcnn_shared/best.keras | 1.49 MB
runs/customcnn_shared/final.keras | 1.49 MB
runs/customcnn_shared/latest.keras | 1.49 MB
runs/mobilenetv2_shared/best.keras | 11.63 MB
runs/mobilenetv2_shared/final.keras | 11.62 MB
runs/mobilenetv2_shared/latest.keras | 11.63 MB

FEATURE ARRAY VALIDATION
features/customcnn_shared/food/emb_128.npy | shape=(10277, 128) | dtype=float32
features/customcnn_shared/food/pooled.npy | shape=(10277, 128) | dtype=float32
features/customcnn_shared/water/emb_128.npy | shape=(617, 128) | dtype=float32
features/customcnn_shared/water/pooled.npy | shape=(617, 128) | dtype=float32
features/mobilenetv2_shared/food/emb_128.npy | shape=(10277, 128) | dtype=float32
features/mobilenetv2_shared/food/pooled.npy | shape=(10277, 1280) | dtype=float32
features/mobilenetv2_shared/water/emb_128.npy | shape=(617, 128) | dtype=float32
features/mobilenetv2_shared/water/pooled.npy | shape=(617, 1280) | dtype=float32

FEATURE IN

In [15]:

path = "/home/kapoo/sharedcnn_project/shared_train.py"

with open(path, "r", encoding="utf-8") as f:
    lines = f.readlines()

for start, end in [(1, 25), (55, 110), (110, 180)]:
    print(f"\n{'='*15} LINES {start}-{end} {'='*15}")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03}: {lines[i-1].rstrip()}")



=============== LINES 1-25 ===============
001: 
002: import os, json, csv, hashlib
003: from pathlib import Path
004: import numpy as np
005: import tensorflow as tf
006: 
007: ROOT = Path.home() / "sharedcnn_project"
008: DATA, OUT = ROOT / "data", ROOT / "runs"
009: OUT.mkdir(exist_ok=True)
010: 
011: SEED = 42
012: IMG = (128, 128)
013: BATCH = 16
014: EPOCHS = 50
015: 
016: tf.keras.utils.set_random_seed(SEED)
017: for g in tf.config.list_physical_devices("GPU"):
018:     tf.config.experimental.set_memory_growth(g, True)
019: 
020: FOOD_DIR, WATER_DIR = DATA / "food", DATA / "water"
021: FOOD_CLASSES = sorted(p.name for p in FOOD_DIR.iterdir() if p.is_dir())
022: WATER_CLASSES = ["Clear", "Foam", "Highly_Turbid", "Medium_Turbid"]
023: 
024: def collect_food():
025:     paths, labels = [], []

=============== LINES 55-110 ===============
055:         seed=SEED,
056:     )
057:     return ds.prefetch(tf.data.AUTOTUNE)
058: 
059: def build_shared(architecture):
060:     inputs = tf.

In [16]:

path = "/home/kapoo/sharedcnn_project/shared_train.py"

with open(path, "r", encoding="utf-8") as f:
    lines = f.readlines()

for start, end in [(24, 58), (127, 165)]:
    print(f"\n{'='*12} LINES {start}-{end} {'='*12}")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03}: {lines[i-1].rstrip()}")



============ LINES 24-58 ============
024: def collect_food():
025:     paths, labels = [], []
026:     for label, cls in enumerate(FOOD_CLASSES):
027:         for p in (FOOD_DIR / cls).rglob("*"):
028:             if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
029:                 paths.append(str(p))
030:                 labels.append(label)
031:     return paths, labels
032: 
033: def load_image(path, label):
034:     image = tf.io.decode_image(
035:         tf.io.read_file(path), channels=3, expand_animations=False
036:     )
037:     image.set_shape([None, None, 3])
038:     image = tf.image.resize(image, IMG)
039:     return image, label
040: 
041: def make_food_ds(paths, labels, training):
042:     ds = tf.data.Dataset.from_tensor_slices((paths, labels))
043:     if training:
044:         ds = ds.shuffle(min(len(paths), 3000), seed=SEED)
045:     return ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE) \
046:              .batch(BATCH).

In [17]:

path = "/home/kapoo/sharedcnn_project/shared_train.py"

with open(path, "r", encoding="utf-8") as f:
    lines = f.readlines()

for start, end in [(24, 58), (127, 165)]:
    print(f"\n{'='*12} LINES {start}-{end} {'='*12}")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03}: {lines[i-1].rstrip()}")



============ LINES 24-58 ============
024: def collect_food():
025:     paths, labels = [], []
026:     for label, cls in enumerate(FOOD_CLASSES):
027:         for p in (FOOD_DIR / cls).rglob("*"):
028:             if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
029:                 paths.append(str(p))
030:                 labels.append(label)
031:     return paths, labels
032: 
033: def load_image(path, label):
034:     image = tf.io.decode_image(
035:         tf.io.read_file(path), channels=3, expand_animations=False
036:     )
037:     image.set_shape([None, None, 3])
038:     image = tf.image.resize(image, IMG)
039:     return image, label
040: 
041: def make_food_ds(paths, labels, training):
042:     ds = tf.data.Dataset.from_tensor_slices((paths, labels))
043:     if training:
044:         ds = ds.shuffle(min(len(paths), 3000), seed=SEED)
045:     return ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE) \
046:              .batch(BATCH).

In [18]:

from pathlib import Path
import shutil

p = Path("/home/kapoo/sharedcnn_project/shared_train.py")
backup = p.with_name("shared_train_backup.py")

if not backup.exists():
    shutil.copy2(p, backup)

lines = p.read_text(encoding="utf-8").splitlines()
# Remove the accidental shell command and stray pasted characters.
while lines and not lines[-1].strip():
    lines.pop()
if lines and lines[-1].strip() == "XXX":
    lines.pop()
if lines and "python -m pip install jupyterlab ipykernel" in lines[-1]:
    lines.pop()
while lines and not lines[-1].strip():
    lines.pop()

p.write_text("\n".join(lines) + "\n", encoding="utf-8")

print("Backup:", backup)
print("Cleaned script:", p)
print("No training was started.")
print("Next we will correct the multi-output targets and balanced batch loop.")


Backup: /home/kapoo/sharedcnn_project/shared_train_backup.py
Cleaned script: /home/kapoo/sharedcnn_project/shared_train.py
No training was started.
Next we will correct the multi-output targets and balanced batch loop.


In [20]:

from pathlib import Path

p = Path("/home/kapoo/sharedcnn_project/shared_train.py")
src = p.read_text(encoding="utf-8", errors="replace")

# Remove the corrupted trailing shell commands and control characters.
src = "".join(ch for ch in src if ch in "\n\r\t" or ord(ch) >= 32)
lines = src.splitlines()

# Remove accidental shell commands if they remain.
lines = [
    line for line in lines
    if "python -m pip install jupyterlab ipykernel" not in line
    and line.strip() not in {"XXX", "XXX"}
]

p.write_text("\n".join(lines).rstrip() + "\n", encoding="utf-8")

# Verify the script parses now.
compile(p.read_text(encoding="utf-8"), str(p), "exec")

print("SUCCESS: shared_train.py is syntactically valid.")
print("Existing checkpoints were not changed.")
print("Backup remains at:", p.with_name("shared_train_backup.py"))


SUCCESS: shared_train.py is syntactically valid.
Existing checkpoints were not changed.
Backup remains at: /home/kapoo/sharedcnn_project/shared_train_backup.py


In [21]:

from pathlib import Path

p = Path("/home/kapoo/sharedcnn_project/shared_train.py")
src = p.read_text(encoding="utf-8")

compile(src, str(p), "exec")
lines = src.splitlines()

print("Script syntax: OK")
print("\nCurrent training loop:")
for i in range(124, min(160, len(lines))):
    print(f"{i+1:03}: {lines[i]}")

print("\nModel compile configuration:")
for i, line in enumerate(lines):
    if "model.compile(" in line:
        for j in range(i, min(i + 16, len(lines))):
            print(f"{j+1:03}: {lines[j]}")
        break

print("\nReady to replace the training logic.")


Script syntax: OK

Current training loop:
125: water_val = make_water_ds("valid", False)
126: 
127: # Separate iterators; shared model trains each domain's own head.
128: # Each domain contributes batches to the same trunk.
129: for architecture in ("mobilenetv2", "customcnn"):
130:     run_dir = OUT / f"{architecture}_shared"
131:     run_dir.mkdir(exist_ok=True)
132: 
133:     model, embedder = build_shared(architecture)
134:     model.summary()
135: 
136:     best_val = float("inf")
137:     patience, wait = 5, 0
138: 
139:     for epoch in range(EPOCHS):
140:         food_metrics = model.fit(
141:             food_train,
142:             validation_data=food_val,
143:             epochs=1,
144:             verbose=1,
145:         )
146:         water_metrics = model.fit(
147:             water_train,
148:             validation_data=water_val,
149:             epochs=1,
150:             verbose=1,
151:         )
152: 
153:         # Save each epoch so interruption doesn't lose all 

In [22]:

from pathlib import Path
import shutil

p = Path("/home/kapoo/sharedcnn_project/shared_train.py")
src = p.read_text(encoding="utf-8")

start = src.index("# Separate iterators; shared model trains each domain's own head.")
end = src.index('print("Shared training loops completed.", flush=True)', start)
end += len('print("Shared training loops completed.", flush=True)')

replacement = r'''
# Correct shared training: alternate domains using explicit task-specific losses.
# Each batch updates the shared trunk and only its relevant classification head.
class SharedTrainer(tf.keras.Model):
    def __init__(self, backbone_model):
        super().__init__()
        self.network = backbone_model

    def train_step(self, data):
        images, targets = data
        with tf.GradientTape() as tape:
            outputs = self.network(images, training=True)
            loss = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    targets["food_head"], outputs["food_head"]
                )
            ) if "food_head" in targets else tf.constant(0.0)

            if "water_head" in targets:
                water_loss = tf.reduce_mean(
                    tf.keras.losses.sparse_categorical_crossentropy(
                        targets["water_head"], outputs["water_head"]
                    )
                )
                loss += water_loss

            if self.network.losses:
                loss += tf.add_n(self.network.losses)

        grads = tape.gradient(loss, self.network.trainable_variables)
        self.optimizer.apply_gradients(
            [(g, v) for g, v in zip(grads, self.network.trainable_variables)
             if g is not None]
        )
        return {"loss": loss}

def food_targets(images, labels):
    return images, {"food_head": labels}

def water_targets(images, labels):
    return images, {"water_head": labels}

food_train_task = food_train.map(food_targets, num_parallel_calls=tf.data.AUTOTUNE)
water_train_task = water_train.map(water_targets, num_parallel_calls=tf.data.AUTOTUNE)

for architecture in ("mobilenetv2", "customcnn"):
    run_dir = OUT / f"{architecture}_shared"
    run_dir.mkdir(exist_ok=True)

    model, embedder = build_shared(architecture)
    trainer = SharedTrainer(model)
    trainer.compile(optimizer=tf.keras.optimizers.Adam(1e-3))

    food_iter = iter(food_train_task.repeat())
    water_iter = iter(water_train_task.repeat())

    for epoch in range(EPOCHS):
        # Equal number of batches from each domain.
        steps = min(
            int(tf.data.experimental.cardinality(food_train).numpy()),
            int(tf.data.experimental.cardinality(water_train).numpy())
        )
        for _ in range(steps):
            trainer.train_step(next(food_iter))
            trainer.train_step(next(water_iter))

        model.save(run_dir / "latest.keras")
        print(f"{architecture}: epoch {epoch+1}/{EPOCHS}", flush=True)

    model.save(run_dir / "final.keras")
    print("Saved:", run_dir, flush=True)

print("Shared training loops completed.", flush=True)
'''

# Preserve the current file if this is the first replacement attempt.
backup = p.with_name("shared_train_before_loop_fix.py")
if not backup.exists():
    shutil.copy2(p, backup)

updated = src[:start] + replacement + "\n"
compile(updated, str(p), "exec")
p.write_text(updated, encoding="utf-8")

print("Training loop replaced and syntax validated.")
print("Backup:", backup)
print("Training has NOT started.")


Training loop replaced and syntax validated.
Backup: /home/kapoo/sharedcnn_project/shared_train_before_loop_fix.py
Training has NOT started.


In [23]:

from pathlib import Path
import os, glob
import tensorflow as tf

ROOT = Path("/home/kapoo/sharedcnn_project")
SCRIPT = ROOT / "shared_train.py"

# 1. Syntax
source = SCRIPT.read_text(encoding="utf-8")
compile(source, str(SCRIPT), "exec")
print("Script syntax: OK")

# 2. Existing checkpoints
print("\nExisting checkpoints:")
for model in ["mobilenetv2_shared", "customcnn_shared"]:
    files = glob.glob(str(ROOT / "runs" / model / "*.keras"))
    print(model, "->", len(files), "checkpoint files")
    for f in files:
        print(" ", os.path.basename(f), f"{os.path.getsize(f)/1e6:.2f} MB")

# 3. Dataset sizes
print("\nDataset batches:")
for name, ds in [
    ("food_train", "food_train"),
    ("water_train", "water_train"),
]:
    print(name, "defined in script:", f"{name} =" in source)

# 4. Ensure no accidental shell commands remain
print("\nAccidental pip command in script:",
      "python -m pip install" in source)
print("No training started.")


Script syntax: OK

Existing checkpoints:
mobilenetv2_shared -> 3 checkpoint files
  best.keras 11.63 MB
  latest.keras 11.63 MB
  final.keras 11.62 MB
customcnn_shared -> 3 checkpoint files
  best.keras 1.49 MB
  latest.keras 1.49 MB
  final.keras 1.49 MB

Dataset batches:
food_train defined in script: True
water_train defined in script: True

Accidental pip command in script: False
No training started.


In [24]:

from pathlib import Path
import ast

ROOT = Path("/home/kapoo/sharedcnn_project")
SCRIPT = ROOT / "shared_train.py"
source = SCRIPT.read_text(encoding="utf-8")
tree = ast.parse(source)

print("=== SCRIPT CONFIGURATION ===")
for line in source.splitlines():
    if any(term in line for term in [
        "SEED =", "IMG =", "BATCH =", "EPOCHS =",
        "food_classes", "water_classes", "Unlabeled",
        "def build_shared", "class SharedTrainer",
        "def train_step", "def food_targets", "def water_targets",
        "def make_food_ds", "def make_water_ds",
        "def collect_food", "classification_report",
        "confusion_matrix", "macro_f1", "EarlyStopping"
    ]):
        print(line[:180])

print("\n=== REQUIRED OUTPUTS ===")
for folder in ["runs", "features", "outputs"]:
    path = ROOT / folder
    print(f"\n{folder}/")
    if path.exists():
        for p in sorted(path.rglob("*")):
            if p.is_file() and (
                p.suffix in {".keras", ".npy", ".csv", ".json", ".txt"}
            ):
                print(f"  {p.relative_to(ROOT)}")
    else:
        print("  MISSING")

print("\n=== DATASET CLASS FOLDERS ===")
for split in ["train", "valid", "test"]:
    folder = ROOT / "data" / "water" / split
    if folder.exists():
        print(split, sorted(p.name for p in folder.iterdir() if p.is_dir()))

print("\nAudit complete — no training started.")


=== SCRIPT CONFIGURATION ===
SEED = 42
IMG = (128, 128)
BATCH = 16
EPOCHS = 50
def collect_food():
def make_food_ds(paths, labels, training):
def make_water_ds(split, training):
def build_shared(architecture):
class SharedTrainer(tf.keras.Model):
    def train_step(self, data):
def food_targets(images, labels):
def water_targets(images, labels):

=== REQUIRED OUTPUTS ===

runs/
  runs/customcnn_shared/best.keras
  runs/customcnn_shared/final.keras
  runs/customcnn_shared/history.json
  runs/customcnn_shared/latest.keras
  runs/mobilenetv2_shared/best.keras
  runs/mobilenetv2_shared/final.keras
  runs/mobilenetv2_shared/history.json
  runs/mobilenetv2_shared/latest.keras

features/
  features/customcnn_shared/food/emb_128.npy
  features/customcnn_shared/food/index.csv
  features/customcnn_shared/food/pooled.npy
  features/customcnn_shared/water/emb_128.npy
  features/customcnn_shared/water/index.csv
  features/customcnn_shared/water/pooled.npy
  features/mobilenetv2_shared/food/emb_128.

In [25]:

import tensorflow as tf
from tensorflow.keras import layers, regularizers
from pathlib import Path

ROOT = Path("/home/kapoo/sharedcnn_project")
RUN_DIR = ROOT / "runs" / "customcnn_v2_shared"
RUN_DIR.mkdir(parents=True, exist_ok=True)

def conv_bn_relu(x, filters, stride=1):
    x = layers.Conv2D(
        filters, 3, strides=stride, padding="same",
        use_bias=False, kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)

def se_block(x, ratio=8):
    channels = int(x.shape[-1])
    scale = layers.GlobalAveragePooling2D()(x)
    scale = layers.Dense(max(channels // ratio, 8),
                         activation="relu")(scale)
    scale = layers.Dense(channels, activation="sigmoid")(scale)
    scale = layers.Reshape((1, 1, channels))(scale)
    return layers.Multiply()([x, scale])

def residual_block(x, filters, stride=1):
    shortcut = x

    y = conv_bn_relu(x, filters, stride)
    y = layers.Conv2D(
        filters, 3, padding="same", use_bias=False,
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4)
    )(y)
    y = layers.BatchNormalization()(y)
    y = se_block(y)

    if stride != 1 or int(shortcut.shape[-1]) != filters:
        shortcut = layers.Conv2D(
            filters, 1, strides=stride, padding="same",
            use_bias=False
        )(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([shortcut, y])
    return layers.Activation("relu")(x)

def build_customcnn_v2():
    inputs = layers.Input((128, 128, 3), name="image")

    x = conv_bn_relu(inputs, 32)
    x = conv_bn_relu(x, 32)

    for filters in (32, 64, 128, 192):
        x = residual_block(x, filters)
        x = residual_block(x, filters, stride=2)

    pooled = layers.GlobalAveragePooling2D(name="pooled")(x)
    x = layers.Dropout(0.30)(pooled)
    embedding = layers.Dense(
        128, activation="relu", name="embedding_128"
    )(x)
    embedding_out = layers.Dropout(0.15)(embedding)

    outputs = {
        "food_head": layers.Dense(
            12, activation="softmax", name="food_head"
        )(embedding_out),
        "water_head": layers.Dense(
            4, activation="softmax", name="water_head"
        )(embedding_out)
    }

    model = tf.keras.Model(inputs, outputs, name="customcnn_v2_shared")
    embedder = tf.keras.Model(
        inputs,
        {"embedding": embedding, "pooled": pooled},
        name="customcnn_v2_embedder"
    )

    return model, embedder

customcnn_v2, customcnn_v2_embedder = build_customcnn_v2()

# Compile only; no training occurs here.
customcnn_v2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss={
        "food_head": "sparse_categorical_crossentropy",
        "water_head": "sparse_categorical_crossentropy"
    },
    metrics={
        "food_head": ["accuracy"],
        "water_head": ["accuracy"]
    }
)

# Verify the new architecture with a dummy input.
dummy = tf.zeros((1, 128, 128, 3))
predictions = customcnn_v2(dummy, training=False)
features = customcnn_v2_embedder(dummy, training=False)

print("Model:", customcnn_v2.name)
print("Food output:", predictions["food_head"].shape)
print("Water output:", predictions["water_head"].shape)
print("Embedding:", features["embedding"].shape)
print("Pooled features:", features["pooled"].shape)
print("Parameters:", f"{customcnn_v2.count_params():,}")
print("New output directory:", RUN_DIR)
print("Architecture verified. No training started.")


Model: customcnn_v2_shared
Food output: (1, 12)
Water output: (1, 4)
Embedding: (1, 128)
Pooled features: (1, 192)
Parameters: 2,069,280
New output directory: /home/kapoo/sharedcnn_project/runs/customcnn_v2_shared
Architecture verified. No training started.


In [ ]:

import json
import numpy as np
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import f1_score

# ============================================================
# CUSTOM CNN V2 — COMPLETE TRAINING
# Requires: customcnn_v2, food_train, food_val,
#           water_train, water_val from earlier notebook cells
# ============================================================

ROOT = Path("/home/kapoo/sharedcnn_project")
RUN_DIR = ROOT / "runs" / "customcnn_v2_shared"
RUN_DIR.mkdir(parents=True, exist_ok=True)

required = [
    "customcnn_v2", "food_train", "food_val",
    "water_train", "water_val"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing: {missing}. Run the model-building and data-loading "
        "cells first. Do not rerun the old training script."
    )

# Initialize ONE optimizer with ALL model variables.
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-4)
optimizer.build(customcnn_v2.trainable_variables)

def prepare_task(dataset, head):
    return dataset.map(
        lambda images, labels: (images, tf.cast(labels, tf.int32)),
        num_parallel_calls=tf.data.AUTOTUNE
    )

food_train_ds = prepare_task(food_train, "food_head").repeat()
water_train_ds = prepare_task(water_train, "water_head").repeat()

food_iter = iter(food_train_ds)
water_iter = iter(water_train_ds)

food_steps = int(tf.data.experimental.cardinality(food_train).numpy())
water_steps = int(tf.data.experimental.cardinality(water_train).numpy())

if food_steps < 1 or water_steps < 1:
    raise RuntimeError(
        f"Dataset cardinality invalid: food={food_steps}, water={water_steps}. "
        "Check that the training datasets are finite and non-empty."
    )

# Alternate task batches. Each batch updates only its own head
# and the shared backbone.
steps_per_epoch = max(food_steps, water_steps)
EPOCHS = 20
history = []
best_score = -1.0

def train_one_batch(images, labels, head):
    with tf.GradientTape() as tape:
        predictions = customcnn_v2(images, training=True)[head]
        loss = tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(
                labels, predictions
            )
        )
        if customcnn_v2.losses:
            loss += tf.add_n(customcnn_v2.losses)

    variables = customcnn_v2.trainable_variables
    gradients = tape.gradient(loss, variables)
    pairs = [
        (g, v) for g, v in zip(gradients, variables)
        if g is not None
    ]
    optimizer.apply_gradients(pairs)
    return float(loss.numpy())

def evaluate(dataset, head):
    true_labels, predicted_labels = [], []

    for images, labels in dataset:
        probs = customcnn_v2(images, training=False)[head]
        true_labels.extend(labels.numpy().astype(int).tolist())
        predicted_labels.extend(
            tf.argmax(probs, axis=1).numpy().astype(int).tolist()
        )

    if not true_labels:
        raise RuntimeError(f"Validation dataset for {head} is empty.")

    return {
        "accuracy": float(np.mean(
            np.asarray(true_labels) == np.asarray(predicted_labels)
        )),
        "macro_f1": float(f1_score(
            true_labels, predicted_labels,
            average="macro", zero_division=0
        ))
    }

print("Starting Custom CNN V2 training...")
print("Food steps:", food_steps, "| Water steps:", water_steps)
print("Saving to:", RUN_DIR)

for epoch in range(EPOCHS):
    losses = []

    for step in range(steps_per_epoch):
        # Balance the number of updates from each domain.
        # The shorter dataset cycles; the longer dataset is covered once.
        for iterator, head in (
            (food_iter, "food_head"),
            (water_iter, "water_head")
        ):
            images, labels = next(iterator)
            losses.append(train_one_batch(images, labels, head))

    food_metrics = evaluate(food_val, "food_head")
    water_metrics = evaluate(water_val, "water_head")

    score = (
        food_metrics["macro_f1"] + water_metrics["macro_f1"]
    ) / 2

    record = {
        "epoch": epoch + 1,
        "train_loss": float(np.mean(losses)),
        "food_val_accuracy": food_metrics["accuracy"],
        "food_val_macro_f1": food_metrics["macro_f1"],
        "water_val_accuracy": water_metrics["accuracy"],
        "water_val_macro_f1": water_metrics["macro_f1"],
        "mean_val_macro_f1": float(score)
    }
    history.append(record)

    customcnn_v2.save(RUN_DIR / "latest.keras")

    if score > best_score:
        best_score = score
        customcnn_v2.save(RUN_DIR / "best.keras")

    with open(RUN_DIR / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"loss {record['train_loss']:.4f} | "
        f"Food acc {food_metrics['accuracy']:.3f}, "
        f"F1 {food_metrics['macro_f1']:.3f} | "
        f"Water acc {water_metrics['accuracy']:.3f}, "
        f"F1 {water_metrics['macro_f1']:.3f}"
    )

customcnn_v2.save(RUN_DIR / "final.keras")

print("\nTraining complete.")
print("Best validation mean macro-F1:", round(best_score, 4))
print("Best checkpoint:", RUN_DIR / "best.keras")
print("Final checkpoint:", RUN_DIR / "final.keras")
print("History:", RUN_DIR / "history.json")


Starting Custom CNN V2 training...
Food steps: 643 | Water steps: 39
Saving to: /home/kapoo/sharedcnn_project/runs/customcnn_v2_shared


W0000 00:00:1792718583.055962   14319 prefetch_dataset_op.cc:476] SEVERE STARVATION: Element UID 1792718572978082228 in buffer 'PrefetchDataset/_14' sat unconsumed for 10.0777 seconds!
W0000 00:00:1792719413.139109   14325 prefetch_dataset_op.cc:476] SEVERE STARVATION: Element UID 1792719402772246620 in buffer 'PrefetchDataset/_16' sat unconsumed for 10.3668 seconds!


Epoch 01/20 | loss 0.8160 | Food acc 0.616, F1 0.532 | Water acc 0.642, F1 0.347


W0000 00:00:1792719585.733962    8760 prefetch_dataset_op.cc:476] SEVERE STARVATION: Element UID 1792719560207317255 in buffer 'inject/prefetch_RepeatDataset/_18' sat unconsumed for 25.5124 seconds!
